# Inventory Feature Engineering
Building the inventory feature table using the precomputed CSVs and raw data.

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('../../data/precomputed', exist_ok=True)

## Load Data
We need specific columns from the raw data to compute reorder rates and basket position.

In [2]:
cols = ['product_name', 'order_dow', 'add_to_cart_order', 'reordered']
df = pd.read_csv('../../data/combined_instacart_data.csv', usecols=cols)
abc_df = pd.read_csv('../../data/precomputed/abc_analysis.csv')

## Compute Base Metrics
Calculate average basket position and reorder rate per product.

In [3]:
product_stats = df.groupby('product_name').agg(
    reorder_rate=('reordered', 'mean'),
    avg_basket_position=('add_to_cart_order', 'mean'),
    total_orders=('product_name', 'count')
).reset_index()

## Demand Velocity
Demand velocity per product per DOW, then average and std demand.

In [4]:
dow_demand = df.groupby(['product_name', 'order_dow']).size().reset_index(name='demand')
# For simplicity we average the demand across the 7 days for the ROP and Safety Stock formula
demand_agg = dow_demand.groupby('product_name').agg(
    avg_demand=('demand', 'mean'),
    std_demand=('demand', 'std')
).reset_index()
# Fill NaNs for std_demand with 0
demand_agg['std_demand'] = demand_agg['std_demand'].fillna(0)

## Inventory Formulas
Safety stock = Z * std_demand * sqrt(lead_time)
Reorder point = avg_demand * lead_time + safety_stock
EOQ = sqrt(2 * D * S / H)

In [5]:
Z = 1.65
lead_time = 2
S = 50
H = 2

inventory = pd.merge(product_stats, demand_agg, on='product_name', how='left')

# Safety Stock
inventory['safety_stock'] = Z * inventory['std_demand'] * np.sqrt(lead_time)

# Reorder Point
inventory['reorder_point'] = inventory['avg_demand'] * lead_time + inventory['safety_stock']

# EOQ (D = annual demand = avg_demand * 365. Let's use 365)
annual_demand = inventory['avg_demand'] * 365
inventory['eoq'] = np.sqrt((2 * annual_demand * S) / H)

# Merge with ABC
inventory_features = pd.merge(inventory, abc_df[['product_name', 'abc_tier']], on='product_name', how='left')

## Export Data
Save the final features to CSV.

In [6]:
import json
inventory_features.to_csv('../../data/precomputed/inventory_features.csv', index=False)
print('Inventory features exported successfully.')

# Create JSON mapping of product -> dow_demand for the frontend lookup table.
# Include all products with DOW demand data for complete coverage
dow_dict = {}
for prod, group in dow_demand.groupby('product_name'):
    # Convert order_dow (int) keys to strings for JSON compatibility
    dow_dict[prod] = {str(dow): int(demand) for dow, demand in group.set_index('order_dow')['demand'].items()}

with open('../../data/precomputed/product_dow_demand.json', 'w') as f:
    json.dump(dow_dict, f)
    
print(f'Product DOW demand JSON exported: {len(dow_dict)} products')

Inventory features exported successfully.
Product DOW demand JSON exported: 49685 products
